# PlantDoc field test

In [37]:
from IPython.display import display

RUN = True
MODE = "screened_report"

HANDOFF_ROOT = "/kaggle/input/models/astitvasrivastava05/terravista-final-handoff/other/default/V1"
SPLIT_CSV = None
LAB_ROOT = "/kaggle/input/datasets/astitvasrivastava05/terravista-lab-test-predictions"
LAB_DUPLICATE_FLAGS = None  # Auto-detect the attached flags dataset. Override with a full CSV path if needed.
FIELD_RESULTS_DIR = "/kaggle/working/plantdoc_field_test"
REPORT_DIR = "/kaggle/working/plantdoc-field-test"

PLANTDOC_ROOT = None
PLANTVILLAGE_COLOR = None
CHECKPOINT_FILES = {}
LAB_FILES = {}


In [38]:
import hashlib
import json
import re
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from scipy.fft import dctn

MODELS = ("resnet50", "mobilenetv3_large")
SEEDS = (42, 123, 2026)
RUNS = [(model, seed) for model in MODELS for seed in SEEDS]
SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
SPLIT_SHA = "902f6a241712a1a37854ceee4b3833768e723288fd3d00e23f17576269aaf95c"

def file_sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def safe_path(value):
    p = PurePosixPath(str(value).replace("\\", "/"))
    if p.is_absolute() or ".." in p.parts or not p.parts:
        raise ValueError(f"Expected a dataset-relative path, got {value!r}")
    return p.as_posix()

def pv_path(value):
    return re.sub(r"^(?:color|colour)/", "", safe_path(value), flags=re.I)

def read_split(path, class_map):
    if file_sha(path) != SPLIT_SHA:
        raise ValueError("Use the exact final split CSV supplied for this project.")
    frame = pd.read_csv(path)
    frame["path"] = frame["path"].map(pv_path)
    frame["split"] = frame["split"].str.lower().str.strip().replace({"validation": "val"})
    frame["y_true"] = frame["class"].map(class_map)
    if frame.path.duplicated().any() or frame.y_true.isna().any() or set(frame.split) != {"train","val","test"}:
        raise ValueError("Unexpected path, class or split in PlantVillage manifest.")
    return frame

def collect_field(root, mapping, class_map):
    frames = []
    for split in ("train", "test"):
        folder = root / split
        if not folder.is_dir():
            raise FileNotFoundError(f"Expected PlantDoc Classification folder: {folder}")
        for path in sorted(folder.rglob("*")):
            if not path.is_file() or path.suffix.lower() not in SUFFIXES:
                continue
            rel = path.relative_to(root)
            if len(rel.parts) < 3:
                raise ValueError(f"Expected split/class/image in PlantDoc: {rel}")
            label = rel.parts[1]
            if label not in mapping:
                raise ValueError(f"PlantDoc folder is absent from the reviewed mapping: {label}")
            frames.append((rel.as_posix(), split, label, class_map[mapping[label]]))
    frame = pd.DataFrame(frames, columns=["path", "split", "source_label", "y_true"])
    if set(frame.split) != {"train","test"}:
        raise ValueError("Both PlantDoc splits must contain images.")
    return frame

def rgb(path, *, orient=False):
    with Image.open(path) as im:
        return (ImageOps.exif_transpose(im) if orient else im).convert("RGB")

def fingerprint(im):
    grey = im.convert("L")
    patch = np.asarray(grey.resize((32, 32), Image.Resampling.LANCZOS), dtype=np.float32)
    low = dctn(patch, norm="ortho")[:8, :8].ravel()[1:]
    grad = np.asarray(grey.resize((9, 8), Image.Resampling.LANCZOS))
    def pack(values):
        return int.from_bytes(np.packbits(values).tobytes(), "big")
    exact = hashlib.sha256(f"{im.width}x{im.height}:".encode() + im.tobytes()).hexdigest()
    return exact, pack(low > np.median(low)), pack(grad[:, 1:] > grad[:, :-1])

def scan(root, paths, name):
    rows = list(paths)
    def read_one(rel):
        try:
            return fingerprint(rgb(root / rel, orient=True)), ""
        except (OSError, ValueError, SyntaxError, Image.DecompressionBombError) as exc:
            return None, str(exc)
    with ThreadPoolExecutor(max_workers=6) as pool:
        result = {}
        for start in range(0, len(rows), 1000):
            chunk = rows[start:start+1000]
            result.update(zip(chunk, pool.map(read_one, chunk)))
            print(f"{name}: {len(result):,}/{len(rows):,} images checked", flush=True)
    return result

class DuplicateIndex:
    """Five disjoint pHash bands retain every candidate within four bit flips."""
    BANDS = ((0, 13), (13, 13), (26, 13), (39, 13), (52, 12))
    LIMIT = 4

    def __init__(self, rows, fingerprints):
        self.entries = {r.path: (r, fingerprints[r.path][0]) for r in rows if fingerprints[r.path][0] is not None}
        self.exact = defaultdict(list)
        self.buckets = defaultdict(list)
        for path, (_, (digest, ph, _)) in self.entries.items():
            self.exact[digest].append(path)
            for band in self.keys(ph):
                self.buckets[band].append(path)

    @classmethod
    def keys(cls, ph):
        return [(offset, (ph >> offset) & ((1 << width) - 1)) for offset, width in cls.BANDS]

    def query(self, fp, own_path=None):
        if fp is None:
            return [], []
        digest, ph, dh = fp
        exact = [p for p in self.exact.get(digest, []) if p != own_path]
        candidates = set().union(*(self.buckets.get(band, ()) for band in self.keys(ph)))
        candidates.difference_update(exact)
        candidates.discard(own_path)
        near = []
        for path in candidates:
            _, other_ph, other_dh = self.entries[path][1]
            pdist, ddist = (ph ^ other_ph).bit_count(), (dh ^ other_dh).bit_count()
            if pdist <= self.LIMIT and ddist <= self.LIMIT:
                near.append((path, pdist, ddist))
        return sorted(exact), sorted(near, key=lambda pair: (pair[1] + pair[2], pair[0]))


def plantvillage_inventory(pv, root):
    known = set(pv.path)
    available = {p.relative_to(root).as_posix() for p in root.rglob("*")
                 if p.suffix.lower() in SUFFIXES and p.is_file()}
    extra = sorted(available - known)
    inventory = pd.concat(
        [pv[["path", "split"]], pd.DataFrame({"path": extra, "split": "outside_split"})],
        ignore_index=True,
    )
    inventory.attrs["missing_manifest_paths"] = sorted(known - available)
    return inventory


def audit_field(field, pv, plantdoc, plantvillage):
    field_fp = scan(plantdoc, field.path, "PlantDoc")
    pv_fp = scan(plantvillage, pv.path, "PlantVillage colour images")
    faults = [(p, error) for p, (fp, error) in pv_fp.items() if fp is None]
    if faults:
        raise ValueError(f"PlantVillage duplicate audit incomplete: {len(faults)} unreadable files. First: {faults[0]}")
    field_index = DuplicateIndex(field.itertuples(index=False), field_fp)
    pv_index = DuplicateIndex(pv.itertuples(index=False), pv_fp)
    duplicate_rows = []

    def record(check, row, index, exact, near, readable, *, internal=False):
        split = lambda p: index.entries[p][0].split
        near_paths = [p for p, _, _ in near]
        duplicate_rows.append((
            check, row.split, row.path, readable, len(exact), len(near),
            sum(split(p) != row.split for p in exact) if internal else None,
            sum(split(p) != row.split for p in near_paths) if internal else None,
            exact[0] if exact else "", near_paths[0] if near_paths else "",
            next((p for p in exact if internal and split(p) != row.split), ""),
            next((p for p in near_paths if internal and split(p) != row.split), ""),
        ))

    for i, row in enumerate(pv.itertuples(index=False), 1):
        exact, near = pv_index.query(pv_fp[row.path][0], row.path)
        record("within_plantvillage", row, pv_index, exact, near, True, internal=True)
        if i % 10000 == 0 or i == len(pv):
            print(f"Within PlantVillage: {i:,}/{len(pv):,} compared", flush=True)

    rows = []
    retained = set()
    # A PlantDoc train image is considered before its test counterpart.
    for row in field.sort_values(["split", "path"], ascending=[False, True]).itertuples(index=False):
        fp, error = field_fp[row.path]
        exact, near = field_index.query(fp, row.path)
        cross_exact, cross_near = pv_index.query(fp)
        record("within_plantdoc", row, field_index, exact, near, fp is not None, internal=True)
        record("plantdoc_vs_plantvillage", row, pv_index, cross_exact, cross_near, fp is not None)
        status, match, ph_dist, dh_dist = "retained", "", None, None
        if fp is None:
            status = "unreadable_plantdoc"
        elif cross_exact:
            status, match = "exact_plantvillage_overlap", cross_exact[0]
            ph_dist = dh_dist = 0
        elif cross_near:
            status = "possible_plantvillage_overlap"
            match, ph_dist, dh_dist = cross_near[0]
        else:
            conflict = next((p for p in exact if field_index.entries[p][0].y_true != row.y_true), None)
            repeated = next((p for p in exact if p in retained), None)
            close = next((item for item in near if item[0] in retained), None)
            if conflict:
                status, match = "conflicting_plantdoc_duplicate", conflict
            elif repeated:
                status, match = "repeated_plantdoc_image", repeated
                ph_dist = dh_dist = 0
            elif close:
                status = "possible_plantdoc_duplicate"
                match, ph_dist, dh_dist = close
            else:
                retained.add(row.path)
        rows.append((row.path, row.split, row.source_label, row.y_true, status, match, ph_dist, dh_dist, error))
    audit = pd.DataFrame(rows, columns=["path","split","source_label","y_true","status","matched_path",
                                        "phash_distance","dhash_distance","read_error"])
    details = pd.DataFrame(duplicate_rows, columns=[
        "check","split","path","readable","exact_matches","near_candidates",
        "exact_matches_other_split","near_candidates_other_split","example_exact_match","example_near_candidate",
        "example_exact_other_split","example_near_other_split",
    ])
    counts = details.assign(
        scanned=details.readable.astype(int),
        unreadable=(~details.readable).astype(int),
        images_with_exact_match=details.exact_matches.gt(0).astype(int),
        images_with_near_candidate=details.near_candidates.gt(0).astype(int),
        exact_match_in_other_split=details.exact_matches_other_split.gt(0).where(details.exact_matches_other_split.notna()),
        near_candidate_in_other_split=details.near_candidates_other_split.gt(0).where(details.near_candidates_other_split.notna()),
    )
    measures = ["scanned","unreadable","images_with_exact_match","images_with_near_candidate",
                "exact_match_in_other_split","near_candidate_in_other_split"]
    combined = counts.assign(split="all")
    summary = pd.concat([counts, combined]).groupby(["check","split"])[measures].sum(min_count=1).reset_index()
    for key in measures:
        summary[key] = summary[key].astype("Int64")
    return audit, details, summary

def resolve_runs(root, overrides, *, predictions=False, extras=()):
    root = Path(root)
    files = []
    for location in (root, *extras):
        if not location.is_dir():
            continue
        for p in location.rglob("*"):
            if not p.is_file():
                continue
            if predictions:
                if p.suffix == ".csv" and "test" in p.stem.lower() and "pred" in p.stem.lower():
                    files.append(p)
            elif p.name in {"best_weights.pt", "best_model.pt", "best_checkpoint.pt", "best_weights.pth"}:
                files.append(p)
    resolved = {}
    missing = []
    for model, seed in RUNS:
        key = f"{model}:{seed}"
        if key in overrides:
            candidates = [Path(overrides[key])]
        else:
            model_token = "resnet50" if model == "resnet50" else "mobilenetv3"
            candidates = []
            for p in files:
                name = p.as_posix().lower()
                compact = re.sub(r"[^a-z0-9]", "", name)
                if model_token in compact and re.search(rf"seed[_ -]?{seed}(?!\d)", name):
                    if not predictions and model == "resnet50":
                        if not re.search(r"frac[_ -]?100(?!\d)", name) and "uniformp010" not in compact:
                            continue
                    if not predictions and model != "resnet50" and "uniformp010" not in compact:
                        continue
                    candidates.append(p)
        if len(candidates) == 1 and candidates[0].is_file():
            resolved[(model, seed)] = candidates[0]
        else:
            missing.append((key, [str(p) for p in candidates]))
    if missing:
        raise FileNotFoundError(f"Missing or ambiguous {'lab test CSVs' if predictions else 'checkpoints'}: {missing}. Supply explicit overrides.")
    if len({p.resolve() for p in resolved.values()}) != 6:
        raise ValueError("Each of the six runs must have its own input file.")
    return resolved

def read_lab(path, expected, class_map):
    frame = pd.read_csv(path)
    keys = {
        "path": ("path", "image_id", "relative_path"),
        "y_true": ("y_true", "true_label", "true_class", "true_id"),
        "y_pred": ("y_pred", "predicted_label", "pred_class", "pred_label", "pred_id"),
    }
    columns = {}
    for target, options in keys.items():
        column = next((name for name in options if name in frame), None)
        if column is None:
            raise ValueError(f"{path.name} needs one of {options} for {target}.")
        columns[target] = frame[column]
    data = pd.DataFrame(columns)
    data["path"] = data.path.map(pv_path)
    def label_id(value):
        return class_map[str(value)] if str(value) in class_map else int(str(value))
    for col in ("y_true", "y_pred"):
        data[col] = data[col].map(label_id)
        if not data[col].between(0, 37).all():
            raise ValueError(f"IDs outside the original 38-class map in {path.name}")
    if data.path.duplicated().any() or set(data.path) != set(expected.path):
        raise ValueError(f"{path.name} must contain exactly one row per official test image (validation CSVs cannot be used).")
    data = data.set_index("path").loc[expected.path].reset_index()
    if not np.array_equal(data.y_true, expected.y_true):
        raise ValueError(f"Ground-truth labels disagree with the official test manifest in {path.name}.")
    return data

def predict_field(path, model_name, seed, cohort, root, class_map, device):
    import torch
    from torchvision.models import mobilenet_v3_large, resnet50
    from torch import nn

    payload = torch.load(path, map_location="cpu", weights_only=True)
    if not isinstance(payload, dict):
        raise ValueError(f"Expected a state-dict checkpoint at {path}")
    state = next((payload[k] for k in ("model_state_dict", "state_dict", "model", "model_state", "weights")
                  if isinstance(payload.get(k), dict)), payload)
    if not all(torch.is_tensor(value) for value in state.values()):
        raise ValueError(f"Unsupported checkpoint format: {path}")
    if model_name == "resnet50" and "config_sha256" in payload:
        if payload["config_sha256"] != EXPECTED_RESNET_CONFIG:
            raise ValueError(f"Checkpoint config does not match the member-provided final ResNet-50 run: {path}")
    if "recipe" in payload and payload["recipe"] != "uniform_p010":
        raise ValueError(f"Expected the final Uniform 0.10 recipe: {path}")
    if "seed" in payload and int(payload["seed"]) != seed:
        raise ValueError(f"Wrong seed in {path}")
    if "class_to_idx" in payload and payload["class_to_idx"] != class_map:
        raise ValueError(f"Class IDs differ in {path}")
    if "fraction" in payload and float(payload["fraction"]) != 1.0:
        raise ValueError(f"Expected full-data checkpoint: {path}")
    if "num_classes" in payload and int(payload["num_classes"]) != 38:
        raise ValueError(f"Expected 38 outputs: {path}")
    if "arch" in payload:
        arch = re.sub(r"[^a-z0-9]", "", str(payload["arch"]).lower())
        expected = {"resnet50"} if model_name == "resnet50" else {"mobilenetv3", "mobilenetv3large"}
        if arch not in expected:
            raise ValueError(f"Unexpected architecture in {path}: {payload['arch']}")
    if all(k.startswith("module.") for k in state):
        state = {k.removeprefix("module."): v for k, v in state.items()}
    if model_name == "resnet50":
        model = resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, 38)
    else:
        model = mobilenet_v3_large(weights=None)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 38)
    model.load_state_dict(state, strict=True)
    model.to(device).eval()
    mean = torch.tensor([.485, .456, .406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([.229, .224, .225], device=device).view(1, 3, 1, 1)
    for key, expected in (("imagenet_mean", [.485, .456, .406]), ("imagenet_std", [.229, .224, .225])):
        if key in payload and not np.allclose(payload[key], expected, atol=1e-6):
            raise ValueError(f"Unexpected preprocessing in {path}")
    predicted, scores = [], []
    with torch.inference_mode():
        for start in range(0, len(cohort), 64):
            paths = cohort.path.iloc[start:start+64]
            images = [np.asarray(rgb(root / p).resize((224, 224), Image.Resampling.BILINEAR)) for p in paths]
            batch = torch.from_numpy(np.stack(images)).to(device).permute(0, 3, 1, 2).float().div_(255)
            logits = model((batch - mean) / std)
            if not torch.isfinite(logits).all().item():
                raise ValueError(f"Non-finite output from {path}")
            score, pred = logits.softmax(dim=1).max(dim=1)
            predicted.extend(pred.cpu().tolist())
            scores.extend(score.cpu().tolist())
    return cohort[["path", "y_true"]].assign(y_pred=predicted, softmax_score=scores)


In [39]:
CLASS_MAP = {
    "Apple___Apple_scab": 0,
    "Apple___Black_rot": 1,
    "Apple___Cedar_apple_rust": 2,
    "Apple___healthy": 3,
    "Blueberry___healthy": 4,
    "Cherry_(including_sour)___Powdery_mildew": 5,
    "Cherry_(including_sour)___healthy": 6,
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": 7,
    "Corn_(maize)___Common_rust_": 8,
    "Corn_(maize)___Northern_Leaf_Blight": 9,
    "Corn_(maize)___healthy": 10,
    "Grape___Black_rot": 11,
    "Grape___Esca_(Black_Measles)": 12,
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)": 13,
    "Grape___healthy": 14,
    "Orange___Haunglongbing_(Citrus_greening)": 15,
    "Peach___Bacterial_spot": 16,
    "Peach___healthy": 17,
    "Pepper,_bell___Bacterial_spot": 18,
    "Pepper,_bell___healthy": 19,
    "Potato___Early_blight": 20,
    "Potato___Late_blight": 21,
    "Potato___healthy": 22,
    "Raspberry___healthy": 23,
    "Soybean___healthy": 24,
    "Squash___Powdery_mildew": 25,
    "Strawberry___Leaf_scorch": 26,
    "Strawberry___healthy": 27,
    "Tomato___Bacterial_spot": 28,
    "Tomato___Early_blight": 29,
    "Tomato___Late_blight": 30,
    "Tomato___Leaf_Mold": 31,
    "Tomato___Septoria_leaf_spot": 32,
    "Tomato___Spider_mites Two-spotted_spider_mite": 33,
    "Tomato___Target_Spot": 34,
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": 35,
    "Tomato___Tomato_mosaic_virus": 36,
    "Tomato___healthy": 37
}
MAPPING = {
    "Apple_Scab_Leaf": "Apple___Apple_scab",
    "Apple_rust_leaf": "Apple___Cedar_apple_rust",
    "Apple_leaf": "Apple___healthy",
    "Blueberry_leaf": "Blueberry___healthy",
    "Cherry_leaf": "Cherry_(including_sour)___healthy",
    "Corn_Gray_leaf_spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_rust_leaf": "Corn_(maize)___Common_rust_",
    "Corn_leaf_blight": "Corn_(maize)___Northern_Leaf_Blight",
    "grape_leaf_black_rot": "Grape___Black_rot",
    "grape_leaf": "Grape___healthy",
    "Peach_leaf": "Peach___healthy",
    "Bell_pepper_leaf_spot": "Pepper,_bell___Bacterial_spot",
    "Bell_pepper_leaf": "Pepper,_bell___healthy",
    "Potato_leaf_early_blight": "Potato___Early_blight",
    "Potato_leaf_late_blight": "Potato___Late_blight",
    "Raspberry_leaf": "Raspberry___healthy",
    "Soyabean_leaf": "Soybean___healthy",
    "Squash_Powdery_mildew_leaf": "Squash___Powdery_mildew",
    "Strawberry_leaf": "Strawberry___healthy",
    "Tomato_leaf_bacterial_spot": "Tomato___Bacterial_spot",
    "Tomato_Early_blight_leaf": "Tomato___Early_blight",
    "Tomato_leaf_late_blight": "Tomato___Late_blight",
    "Tomato_mold_leaf": "Tomato___Leaf_Mold",
    "Tomato_Septoria_leaf_spot": "Tomato___Septoria_leaf_spot",
    "Tomato_two_spotted_spider_mites_leaf": "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato_leaf_yellow_virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato_leaf_mosaic_virus": "Tomato___Tomato_mosaic_virus",
    "Tomato_leaf": "Tomato___healthy"
}
EXPECTED_RESNET_CONFIG = "0883abc5dc11dbc76468f070cbae821d89203e17e91ea850d1e3a34bd08d829f"

def find_folders(names, predicate):
    candidates = []
    for base in names:
        level = [base]
        for depth in range(4):
            candidates.extend(p for p in level if predicate(p))
            if depth < 3:
                level = [d for p in level if not predicate(p) for d in p.iterdir() if d.is_dir()
                         and not (p / "Apple___Apple_scab").is_dir()]
    return list(dict.fromkeys(candidates))

def locate(names, predicate):
    value = find_folders(names, predicate)
    if len(value) != 1:
        raise ValueError(f"Set the matching explicit folder path. Found: {value}")
    return value[0]

def select_plantvillage(mounts, pv, configured=None):
    names = [p for p in mounts if "plantvillage" in p.name.lower()]
    roots = [Path(configured)] if configured else find_folders(
        names, lambda p: p.name.lower() in {"color", "colour"} and (p / "Apple___Apple_scab").is_dir())
    required = set(pv.path)
    valid = []
    incomplete = []
    for root in roots:
        if not root.is_dir():
            incomplete.append(f"{root}: folder is missing")
            continue
        inventory = plantvillage_inventory(pv, root)
        # The inventory includes every manifest path. Check membership against the
        # actual files discovered on disk before using the root for the audit.
        missing = inventory.attrs["missing_manifest_paths"]
        if missing:
            incomplete.append(f"{root}: {len(missing):,} missing split files (e.g. {missing[0]})")
        else:
            valid.append((root, inventory))
    if len(valid) == 1:
        return valid[0]
    if valid:
        official = [(root, inventory) for root, inventory in valid if "abdallahalidev" in root.parts]
        direct = [(root, inventory) for root, inventory in official
                  if root.parent.name == "plantvillage-dataset"]
        if len(direct) == 1:
            print("Both PlantVillage layouts cover the full split. Using the top-level color folder.", flush=True)
            return direct[0]
        raise ValueError(f"Multiple complete PlantVillage roots found: {[str(root) for root, _ in valid]}. Set PLANTVILLAGE_COLOR.")
    raise FileNotFoundError(f"No PlantVillage color folder contains all {len(required):,} split paths. "
                            f"Set PLANTVILLAGE_COLOR. Checked: {incomplete}")

def get_mounts():
    root = Path("/kaggle/input")
    old = [p for p in root.iterdir() if p.is_dir() and p.name != "datasets"]
    datasets = root / "datasets"
    return old + ([p for owner in datasets.iterdir() if owner.is_dir()
                   for p in owner.iterdir() if p.is_dir()] if datasets.is_dir() else [])

def locate_handoff(configured, input_root=Path("/kaggle/input")):
    path = Path(configured)
    if path.is_dir():
        return path
    known = input_root / "models" / "astitvasrivastava05" / "terravista-final-handoff" / "other" / "default"
    if path.parent == known and path.name.lower() in {"1", "v1"}:
        for root in (known, input_root / "astitvasrivastava05" / "terravista-final-handoff" / "other" / "default"):
            for version in ("1", "V1", "v1"):
                candidate = root / version
                if candidate.is_dir():
                    return candidate
    raise FileNotFoundError(
        f"Handoff folder not mounted at {path}. In Kaggle Inputs, expand terravista-final-handoff V1 "
        "and copy the folder path containing the two checkpoint folders and split (1).csv into HANDOFF_ROOT."
    )

def materialize_checkpoints(root):
    from shutil import copyfileobj
    from zipfile import ZipFile
    expanded = []
    for archive in sorted(root.rglob("*.zip")):
        if "checkpoint" not in archive.stem.lower() and "weight" not in archive.stem.lower():
            continue
        target = Path("/kaggle/working/_terravista_weights") / file_sha(archive)[:16]
        with ZipFile(archive) as z:
            for item in z.infolist():
                if PurePosixPath(item.filename).name not in {"best_weights.pt", "best_model.pt", "best_checkpoint.pt", "best_weights.pth"}:
                    continue
                dest = target / safe_path(item.filename)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists() or dest.stat().st_size != item.file_size:
                    tmp = dest.with_suffix(".tmp")
                    with z.open(item) as source, tmp.open("wb") as sink:
                        copyfileobj(source, sink)
                    tmp.replace(dest)
        expanded.append(target)
    return expanded

def score_stack(y_true, predictions, labels, *, samples=4000, rng_seed=921):
    y = np.asarray(y_true, dtype=np.int64)
    p = np.asarray(predictions, dtype=np.int64)
    if p.ndim != 2 or p.shape[1] != len(y) or not len(y):
        raise ValueError("Predictions must align with one common cohort.")
    labels = np.asarray(labels, dtype=np.int64)
    strata = [np.flatnonzero(y == label) for label in labels]
    if not strata or any(not len(g) for g in strata) or not np.isin(y, labels).all():
        raise ValueError("Each scored class must have true support; true labels must be in scored classes.")
    order = np.concatenate(strata)
    p = p[:, order]
    truth = y[order]
    sizes = np.array([len(g) for g in strata])
    offsets = np.r_[0, sizes.cumsum()[:-1]]
    rng = np.random.default_rng(rng_seed)
    def f1(index):
        true = truth[index]
        groups = p[:, index]
        scores = []
        for pred in groups:
            correct = pred == true
            tp = np.bincount(true[correct], minlength=38)[labels]
            predicted = np.bincount(pred, minlength=38)[labels]
            scores.append(np.mean(2 * tp / (sizes + predicted)))
        return scores
    point = np.array(f1(np.arange(len(y))))
    reps = np.empty((samples, p.shape[0]))
    for i in range(samples):
        index = np.concatenate([rng.integers(lo, lo+n, size=n) for lo, n in zip(offsets, sizes)])
        reps[i] = f1(index)
    return point, reps

def build_outputs(field_predictions, lab_predictions, cohort, audit, class_map, out, duplicate_summary):
    from sklearn.metrics import precision_recall_fscore_support
    inv = {idx: label for label, idx in class_map.items()}
    labels = sorted(set(cohort.y_true))
    datasets = {
        "plantdoc_test": (cohort.y_true.to_numpy(), [field_predictions[k].y_pred.to_numpy() for k in RUNS])
    }
    if lab_predictions is not None:
        kept_lab = lab_predictions[RUNS[0]].y_true.isin(labels).to_numpy()
        datasets["plantvillage_test_same_classes"] = (
            lab_predictions[RUNS[0]].y_true.to_numpy()[kept_lab],
            [lab_predictions[k].y_pred.to_numpy()[kept_lab] for k in RUNS],
        )
    summary, seed_rows, class_rows, confused = [], [], [], []
    scores = {}
    for domain, (truth, predicted) in datasets.items():
        point, replicates = score_stack(truth, predicted, labels, rng_seed=921 if domain == "plantdoc_test" else 922)
        scores[domain] = (point, replicates)
        for i, (model, seed) in enumerate(RUNS):
            seed_rows.append((domain, model, seed, point[i]))
            precision, recall, f1, support = precision_recall_fscore_support(
                truth, predicted[i], labels=labels, zero_division=0)
            for j, label in enumerate(labels):
                class_rows.append((domain, model, seed, label, inv[label], int(support[j]), precision[j], recall[j], f1[j]))
            pred = np.asarray(predicted[i])
            pairs = pd.DataFrame({"true_id": truth[pred != truth], "pred_id": pred[pred != truth]})
            for (a,b), count in pairs.groupby(["true_id","pred_id"]).size().items():
                confused.append((domain, model, seed, a, b, inv[a], inv[b], count))
        for m, model in enumerate(MODELS):
            idx = slice(3*m, 3*m+3)
            lo, hi = np.quantile(replicates[:, idx].mean(axis=1), [.025,.975])
            summary.append((domain, model, len(labels), len(truth), point[idx].mean(), point[idx].std(ddof=1), lo, hi))

    results = pd.DataFrame(summary, columns=["domain","model","classes","images","macro_f1_mean","seed_sd","ci_low","ci_high"])
    results.to_csv(out / ("field_metrics.csv" if lab_predictions is None else "field_lab_metrics.csv"), index=False)
    pd.DataFrame(seed_rows, columns=["domain","model","seed","macro_f1"]).to_csv(out / "per_seed_metrics.csv", index=False)
    report = pd.DataFrame(class_rows, columns=["domain","model","seed","class_id","class_label","images","precision","recall","f1"])
    report.to_csv(out / "per_class_metrics.csv", index=False)
    confusions = pd.DataFrame(confused, columns=["domain","model","seed","true_id","pred_id","true_class","pred_class","errors"])
    confusions.to_csv(out / "confusions.csv", index=False)
    support = report.drop_duplicates(["domain","class_id"]).pivot(index=["class_id","class_label"], columns="domain", values="images")
    support.to_csv(out / "images_per_class.csv")
    audit.to_csv(out / "overlap_audit.csv", index=False)
    duplicate_summary.to_csv(out / "duplicate_summary.csv", index=False)

    def interval(values):
        lo, hi = np.quantile(values, [.025,.975])
        return f"[{lo:.4f}, {hi:.4f}]"
    field, field_rep = scores["plantdoc_test"]
    results_index = results.set_index(["domain","model"])
    def metric(domain, model):
        row = results_index.loc[(domain, model)]
        return f"{row.macro_f1_mean:.4f} ± {row.seed_sd:.4f} [{row.ci_low:.4f}, {row.ci_high:.4f}]"
    lines = ["# TerraVista PlantDoc field evaluation", ""]
    if lab_predictions is None:
        lines += ["| Model | PlantDoc Macro-F1 |", "|---|---:|"]
        lines += [f"| {model} | {metric('plantdoc_test', model)} |" for model in MODELS]
        lines += ["", f"Screened PlantDoc test cohort contains {len(cohort)} images in {len(labels)} classes.",
                  "The same-class PlantVillage comparison is pending Member 3's saved test predictions."]
    else:
        lab, lab_rep = scores["plantvillage_test_same_classes"]
        lines += ["| Model | Lab Macro-F1 | Field Macro-F1 | Field minus lab (95% CI) |",
                  "|---|---:|---:|---:|"]
        for m, model in enumerate(MODELS):
            idx = slice(3*m, 3*m+3)
            delta = field[idx].mean() - lab[idx].mean()
            delta_rep = field_rep[:,idx].mean(axis=1)-lab_rep[:,idx].mean(axis=1)
            lines.append(f"| {model} | {metric('plantvillage_test_same_classes', model)} | "
                         f"{metric('plantdoc_test', model)} | {delta:+.4f} {interval(delta_rep)} |")
        lines += ["", f"Both domains are scored on the same {len(labels)} target classes. "
                  f"Field images: {len(cohort)}. Lab images: {int(kept_lab.sum())}. "
                  "The lab cohort uses the official test prediction files filtered to the field-supported classes."]
    paired_diff = field_rep[:,3:].mean(axis=1)-field_rep[:,:3].mean(axis=1)
    lines += ["", f"MobileNetV3-Large minus ResNet-50 on PlantDoc: {field[3:].mean()-field[:3].mean():+.4f} "
              f"{interval(paired_diff)} (paired image bootstrap).",
              "", "Macro-F1 averages equally over true classes in the cohort. Predictions can use all 38 outputs. "
              "SD is the sample standard deviation over the three seeds. The 95% percentile CIs resample images "
              "within true class 4,000 times and share each resample across all six runs in a domain. "
              "The intervals condition on the fitted seeds. For lab vs. field, each domain is resampled independently.",
              "", "## Overlap and exclusions", ""]
    for (split, status), count in audit.groupby(["split","status"]).size().items():
        lines.append(f"- PlantDoc {split}, `{status}`: {count} images.")
    lines += ["", "| Check | Images scanned | Images with exact match | Images with near-duplicate candidate |",
              "|---|---:|---:|---:|"]
    for row in duplicate_summary.loc[duplicate_summary["split"].eq("all")].itertuples(index=False):
        lines.append(f"| {row.check} | {row.scanned} | {row.images_with_exact_match} | {row.images_with_near_candidate} |")
    lines += ["", "Cross-dataset counts are the number of PlantDoc images with a PlantVillage match. "
              "Within-dataset counts include both members of a pair and may count an image in both columns. "
              "The audit checks exact decoded pixels and flags candidates when pHash and dHash distances are both at most four. "
              "Crops or substantial edits can escape screening. See `duplicate_summary.csv` for split-level counts and "
              "`duplicate_checks.csv` for per-image evidence. The field cohort excludes cross-dataset matches and "
              "retains at most one image per detected internal duplicate pair. The official PlantVillage split and saved lab predictions stay fixed.",
              "", "## Measured field failures", ""]
    for model in MODELS:
        subset = report[(report.domain == "plantdoc_test") & (report.model == model)]
        weakest = subset.groupby(["class_id","class_label"]).agg(f1=("f1","mean"), images=("images","first")).nsmallest(3, "f1")
        lines += [f"**{model}**", ""]
        for (_, label), row in weakest.iterrows():
            lines.append(f"- {label}: mean class F1 {row.f1:.3f} ({int(row.images)} images).")
        pairs = confusions[(confusions.domain == "plantdoc_test") & (confusions.model == model)]
        for (truth, pred), count in pairs.groupby(["true_class","pred_class"]).errors.sum().nlargest(3).items():
            lines.append(f"- {truth} predicted as {pred}: {count} errors summed over three seeds.")
        lines.append("")
    lines.append("The confusion counts describe observed errors. Inspect the matching image paths in the prediction CSVs before attributing errors to visual causes.")
    (out / "README.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return results


In [40]:
LAB_DUPLICATE_FLAGS_SHA256 = "d34858415df29c7c3e98d1124574f69d1d6679e44e6f09920cc9ea0073cd9fbc"


def resolve_lab_duplicate_flags(lab_root, explicit_path=None, input_root=Path("/kaggle/input")):
    if explicit_path:
        path = Path(explicit_path)
        if not path.is_file():
            raise FileNotFoundError(f"LAB_DUPLICATE_FLAGS points to a missing file: {path}")
        if file_sha(path) != LAB_DUPLICATE_FLAGS_SHA256:
            raise ValueError(f"LAB_DUPLICATE_FLAGS does not match Member 3's verified handoff file: {path}")
        return path

    roots = [lab_root]
    if input_root.is_dir():
        dataset_names = {"test-near-duplicate-flags", "terravista-lab-test-predictions"}
        mounts = list(input_root.iterdir()) + list(input_root.glob("datasets/*/*"))
        roots.extend(path for path in mounts if path.name.lower().replace("_", "-") in dataset_names)

    candidates = sorted({
        path for root in roots if root.is_dir()
        for path in root.rglob("test_near_duplicate_flags.csv") if path.is_file()
    })
    for path in candidates:
        if file_sha(path) == LAB_DUPLICATE_FLAGS_SHA256:
            return path
    if candidates:
        raise ValueError(f"Found flags CSVs, but none matches Member 3's verified handoff file: {candidates}")
    raise FileNotFoundError(
        "Attach Member 3's test_near_duplicate_flags.csv. If the dataset name differs "
        "from test_near_duplicate_flags, set LAB_DUPLICATE_FLAGS to the full mounted path."
    )


import io


def read_csv(payload):
    return pd.read_csv(io.BytesIO(payload))


def digest(payload):
    return hashlib.sha256(payload).hexdigest()


def require(condition, message):
    if not condition:
        raise ValueError(message)


def align_screening_rows(frame, truth):
    frame = frame.copy()
    frame["path"] = frame.path.map(pv_path)
    require(frame.path.is_unique and set(frame.path) == set(truth.path), "Audit paths differ from the official test split.")
    return frame.set_index("path").loc[truth.path].reset_index()


def to_csv(frame):
    return frame.to_csv(index=False).encode()


def build_screened_report(report_dir, truth, field, predictions, lab, flag_path, provenance):
    mapping = provenance["class_to_idx"]
    source = {name: (report_dir / name).read_bytes() for name in ("duplicate_checks.csv", "field_lab_metrics.csv", "README.md")}
    labels = sorted(field.y_true.unique())
    require(len(truth) == 7978 and len(field) == 241 and len(labels) == 27, "Unexpected input cohort.")
    flag_payload = flag_path.read_bytes()
    flags = align_screening_rows(read_csv(flag_payload), truth)
    require(flags.true_class.equals(truth["class"]), "Flag labels disagree with the split.")
    for name in ("flag_cosine_primary", "flag_hash", "flag_either"):
        require(flags[name].dtype == bool, f"Expected boolean column {name}.")
    require(flags.flag_either.equals(flags.flag_cosine_primary | flags.flag_hash), "Conflicting Member 3 flags.")
    require(int(flags.flag_either.sum()) == 393, "Unexpected Member 3 audit counts.")
    audit = align_screening_rows(read_csv(source["duplicate_checks.csv"]).query('check == "within_plantvillage" and split == "test"'), truth)
    require(audit.readable.eq(True).all(), "Unreadable lab test image.")
    require(audit[["exact_matches", "near_candidates"]].ge(0).all().all(), "Invalid own-audit counts.")
    scored = truth.y_true.isin(labels)
    flagged = flags.flag_either | audit.exact_matches.gt(0) | audit.near_candidates.gt(0)
    kept = scored & ~flagged
    require(set(truth.loc[kept, "y_true"]) == set(labels), "Lab screening removed a field class.")
    excluded = flags.loc[scored & flagged].merge(
        audit[["path", "exact_matches", "near_candidates", "example_exact_match", "example_near_candidate"]],
        on="path", validate="one_to_one",
    )
    require(len(excluded) == int(scored.sum() - kept.sum()), "Missing exclusion evidence.")
    lab_vectors = [lab[key].y_pred.to_numpy() for key in RUNS]
    field_vectors = [predictions[key].y_pred.to_numpy() for key in RUNS]

    field_point, field_draws = score_stack(field.y_true, field_vectors, labels, rng_seed=921)
    lab_point, lab_draws = score_stack(truth.y_true[kept], np.asarray(lab_vectors)[:, kept], labels, rng_seed=922)
    original_point, _ = score_stack(truth.y_true[scored], np.asarray(lab_vectors)[:, scored], labels, rng_seed=922, samples=1)
    original = read_csv(source["field_lab_metrics.csv"]).set_index(["domain", "model"])
    rows, table, seeds = [], [], []
    for model_index, model in enumerate(MODELS):
        columns = slice(model_index * 3, (model_index + 1) * 3)
        for domain, point, draws, n in (
            ("plantdoc_test", field_point, field_draws, len(field)),
            ("plantvillage_test_same_classes_duplicate_screened", lab_point, lab_draws, int(kept.sum())),
        ):
            scores = point[columns]
            replicates = draws[:, columns].mean(axis=1)
            lo, hi = np.percentile(replicates, [2.5, 97.5])
            rows.append((domain, model, len(labels), n, scores.mean(), scores.std(ddof=1), lo, hi))
            seeds.extend((domain, model, seed, f1) for seed, f1 in zip(SEEDS, scores))
        field_row, lab_row = rows[-2:]
        require(np.allclose(field_row[4:], original.loc[("plantdoc_test", model)].iloc[-4:].to_numpy(dtype=float)), "Field scores differ from original.")
        require(np.isclose(original_point[columns].mean(), original.loc[("plantvillage_test_same_classes", model), "macro_f1_mean"]), "Saved predictions do not reproduce original lab score.")
        gap = field_row[4] - lab_row[4]
        lo, hi = np.percentile((field_draws[:, columns] - lab_draws[:, columns]).mean(axis=1), [2.5, 97.5])
        name = {"resnet50": "ResNet-50", "mobilenetv3_large": "MobileNetV3-Large"}[model]
        table.append(f"| {name} | " + " | ".join(f"{row[4]:.4f} ± {row[5]:.4f} [{row[6]:.4f}, {row[7]:.4f}]" for row in (lab_row, field_row)) + f" | {gap:.4f} [{lo:.4f}, {hi:.4f}] |")
    counts = (int(scored.sum()), int((scored & flagged).sum()), int(kept.sum()))
    metadata = {
        "official_lab_test_images": len(truth),
        "official_lab_test_flagged_by_union": int(flagged.sum()),
        "same_class_lab_images_before_screening": counts[0],
        "same_class_lab_images_excluded": counts[1],
        "same_class_lab_images_retained": counts[2],
        "plantdoc_test_images_retained": len(field),
        "scored_class_ids": list(map(int, labels)),
        "lab_exclusion_rule": "Any Member 3 cosine or dHash flag against training, any exact pixel match or dual perceptual hash candidate in Part 4 within-PlantVillage audit",
        "member3_flag_csv_sha256": digest(flag_payload),
        "part4_duplicate_checks_sha256": digest(source["duplicate_checks.csv"]),
        "bootstrap_draws": 4000,
        "bootstrap_rng": {"field": 921, "lab": 922},
    }
    compare_support = pd.DataFrame({
        "class_id": labels,
        "class_name": [{idx: name for name, idx in mapping.items()}[i] for i in labels],
        "plantdoc_test_screened": field.y_true.value_counts().reindex(labels).to_numpy(),
        "plantvillage_test_before_screening": truth.y_true[scored].value_counts().reindex(labels).to_numpy(),
        "plantvillage_test_screened": truth.y_true[kept].value_counts().reindex(labels).to_numpy(),
    })
    source["duplicate_screened_metrics.csv"] = to_csv(pd.DataFrame(rows, columns="domain model classes images macro_f1_mean seed_sd ci_low ci_high".split()))
    source["duplicate_screened_per_seed_metrics.csv"] = to_csv(pd.DataFrame(seeds, columns="domain model seed macro_f1".split()))
    source["duplicate_screened_images_per_class.csv"] = to_csv(compare_support)
    source["lab_duplicate_exclusions.csv"] = to_csv(excluded)
    source["duplicate_screening.json"] = (json.dumps(metadata, indent=2) + "\n").encode()
    old = source["README.md"].decode().split("\n", 1)[1]
    old = old.replace("The field cohort excludes cross-dataset matches and retains at most one image per detected internal duplicate pair.", "The field cohort excluded 11 of 252 test images under the policies listed above. The cross-dataset screen flagged zero images.")
    readme = [
        "# TerraVista PlantDoc field evaluation",
        "",
        "## Main comparison with duplicate screening",
        "",
        "| Model | Screened lab macro-F1 | Screened field macro-F1 | Field minus lab (95% CI) |",
        "|---|---:|---:|---:|",
        *table,
        "",
        f"Both sides cover the same 27 true classes. The main comparison uses {len(field)} PlantDoc test images after excluding 11 flagged images and {counts[2]:,} PlantVillage test images after excluding {counts[1]:,} of {counts[0]:,} same-class images. The cross-dataset scan found zero exact matches and zero near-duplicate candidates under its rules.",
        "",
        "The lab exclusions combine Member 3's cosine similarity ≥ 0.95 or dHash distance ≤ 4 flags against training with Part 4's within-PlantVillage exact pixel matches and dual pHash/dHash candidates. Candidates are exclusions under a conservative policy. Similar images may be flagged, and substantial edits may evade the audit. Only the reporting cohort changes. All scores use the original six saved predictions. The official PlantVillage test results are preserved.",
        "",
        "See `duplicate_screened_metrics.csv` for the table values, `duplicate_screened_images_per_class.csv` for support in both domains, and `lab_duplicate_exclusions.csv` for the flagged lab paths and evidence. The bootstrap and seed summary follow the method below. To reproduce this comparison, run the companion notebook in screened_report mode with the original split, the saved field outputs, Member 3’s six test prediction CSVs and test_near_duplicate_flags.csv.",
        "",
        "## Original official-split lab reference",
        "",
        "The lab numbers in this section precede duplicate screening. The field numbers already use the screened cohort.",
        "",
        old.lstrip("\n"),
    ]
    source["README.md"] = "\n".join(readme).encode()
    for name in (
        "duplicate_screened_metrics.csv", "duplicate_screened_per_seed_metrics.csv",
        "duplicate_screened_images_per_class.csv", "lab_duplicate_exclusions.csv",
        "duplicate_screening.json", "README.md",
    ):
        (report_dir / name).write_bytes(source[name])
    print(f"Screened lab images retained {counts[2]:,} of {counts[0]:,}; PlantDoc images retained {len(field)}.")
    return read_csv(source["duplicate_screened_metrics.csv"])


def write_json(path, data):
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(data, indent=2) + "\n", encoding="utf-8")
    temp.replace(path)

def prediction_filename(model, seed):
    return f"plantdoc_{model}_seed{seed}_predictions.csv"

def run_evaluation():
    if MODE not in {"audit", "field", "report", "screened_report"}:
        raise ValueError("MODE must be audit, field, report or screened_report.")
    handoff = locate_handoff(HANDOFF_ROOT) if MODE == "field" or not SPLIT_CSV else Path(HANDOFF_ROOT)
    if handoff.is_dir():
        print(f"Handoff: {handoff}", flush=True)
    source = Path(FIELD_RESULTS_DIR)
    report_dir = Path(REPORT_DIR)
    if MODE in {"audit", "field"}:
        mounts = get_mounts() if Path("/kaggle/input").is_dir() else []
        plantdoc = Path(PLANTDOC_ROOT) if PLANTDOC_ROOT else locate(
            [p for p in mounts if "plantdoc" in p.name.lower()],
            lambda p: (p / "train").is_dir() and (p / "test").is_dir()
            and not (p / "test" / "images").is_dir())
    if SPLIT_CSV:
        split_path = Path(SPLIT_CSV)
    else:
        candidates = [p for p in handoff.rglob("*.csv") if "split" in p.stem.lower() and file_sha(p) == SPLIT_SHA]
        if not candidates:
            raise FileNotFoundError(f"No matching original split CSV under {handoff}. Keep split (1).csv unchanged or set SPLIT_CSV.")
        split_path = sorted(candidates)[0]
    pv = read_split(split_path, CLASS_MAP)
    print(f"Verified split: {split_path}", flush=True)
    if MODE in {"audit", "field"}:
        plantvillage, pv_images = select_plantvillage(mounts, pv, PLANTVILLAGE_COLOR)
        print(f"PlantVillage: {plantvillage}\nPlantDoc: {plantdoc}", flush=True)
    expected_lab = pv.loc[pv.split.eq("test")].reset_index(drop=True)
    audit_path = source / "overlap_audit.csv"
    audit_meta_path = source / "overlap_audit_provenance.json"
    field_meta_path = source / "field_provenance.json"
    cohort_path = source / "plantdoc_test_cohort.csv"
    duplicate_path = source / "duplicate_checks.csv"
    summary_path = source / "duplicate_summary.csv"
    source_paths = [source / prediction_filename(*key) for key in RUNS]

    if MODE in {"report", "screened_report"}:
        if source.resolve() == report_dir.resolve():
            raise ValueError("Use a separate REPORT_DIR for the combined report.")
        if not field_meta_path.is_file():
            raise FileNotFoundError("Attach the completed field-stage output and point FIELD_RESULTS_DIR to it.")
        provenance = json.loads(field_meta_path.read_text(encoding="utf-8"))
        for name, digest in provenance["field_output_sha256"].items():
            if file_sha(source / name) != digest:
                raise ValueError(f"Field output differs from recorded run: {name}")
        if provenance["split_sha256"] != file_sha(split_path) or provenance["class_to_idx"] != CLASS_MAP:
            raise ValueError("Field outputs came from a different split or model class order.")
        if provenance["mapping"] != MAPPING:
            raise ValueError("Field outputs came from a different PlantDoc label mapping.")
        audit = pd.read_csv(audit_path)
        duplicate_summary = pd.read_csv(summary_path)
        cohort = pd.read_csv(cohort_path)
        predictions = {key: read_field_prediction(path, cohort) for key, path in zip(RUNS, source_paths)}
        lab_files = resolve_runs(Path(LAB_ROOT) if LAB_ROOT else handoff, LAB_FILES, predictions=True)
        lab = {key: read_lab(path, expected_lab, CLASS_MAP) for key, path in lab_files.items()}
        if MODE == "screened_report":
            flag_path = resolve_lab_duplicate_flags(
                Path(LAB_ROOT) if LAB_ROOT else handoff, LAB_DUPLICATE_FLAGS
            )
            print(f"Verified lab duplicate flags: {flag_path}", flush=True)
        report_dir.mkdir(parents=True, exist_ok=True)
        results = build_outputs(predictions, lab, cohort, audit, CLASS_MAP, report_dir, duplicate_summary)
        provenance["lab_test_predictions"] = [
            {"model": model, "seed": seed, "path": str(lab_files[(model, seed)]),
             "sha256": file_sha(lab_files[(model, seed)])} for model, seed in RUNS
        ]
        provenance["plantvillage_test_predictions_per_run"] = len(expected_lab)
        write_json(report_dir / "provenance.json", provenance)
        for path in source_paths + [cohort_path, duplicate_path, summary_path]:
            (report_dir / path.name).write_bytes(path.read_bytes())
        if MODE == "screened_report":
            results = build_screened_report(
                report_dir, expected_lab, cohort, predictions, lab, flag_path, provenance
            )
        display(results)
        print("Commit the notebook and files from", report_dir)
        return

    if MODE == "field" and field_meta_path.is_file():
        raise FileExistsError("This output directory already has a completed field run. Use MODE='report' to reuse it, or choose a new directory.")
    if MODE == "field":
        roots = materialize_checkpoints(handoff)
        weights = resolve_runs(handoff, CHECKPOINT_FILES, extras=roots)
        for (model, seed), path in weights.items():
            if not path.stat().st_size:
                raise ValueError(f"Checkpoint is empty: {path}")
            print(f"{model}, seed {seed}: {path}", flush=True)
        weight_hashes = {key: file_sha(path) for key, path in weights.items()}
        if len(set(weight_hashes.values())) != len(RUNS):
            raise ValueError("The same checkpoint contents were supplied for more than one run.")
    field = collect_field(plantdoc, MAPPING, CLASS_MAP)
    audit_spec = {
        "split_sha256": file_sha(split_path),
        "plantvillage_root": str(plantvillage.resolve()),
        "plantdoc_root": str(plantdoc.resolve()),
        "field_inventory_sha256": hashlib.sha256(field.to_csv(index=False).encode()).hexdigest(),
        "plantvillage_inventory_sha256": hashlib.sha256(pv_images.to_csv(index=False).encode()).hexdigest(),
        "method": "exif_rgb_pixels_phash4_dhash4_all_directions_v2",
    }
    source.mkdir(parents=True, exist_ok=True)
    audit_files = (audit_path, duplicate_path, summary_path)
    if audit_meta_path.is_file():
        saved = json.loads(audit_meta_path.read_text(encoding="utf-8"))
        if saved != {**audit_spec, "files": {p.name: file_sha(p) for p in audit_files}}:
            raise ValueError("Audit inputs differ. Use a fresh FIELD_RESULTS_DIR.")
        audit, duplicate_summary = pd.read_csv(audit_path), pd.read_csv(summary_path)
        print("Reusing the screened duplicate audit.")
    else:
        audit, duplicate_details, duplicate_summary = audit_field(field, pv_images, plantdoc, plantvillage)
        for path, frame in zip(audit_files, (audit, duplicate_details, duplicate_summary)):
            frame.to_csv(path, index=False)
        write_json(audit_meta_path, {**audit_spec, "files": {p.name: file_sha(p) for p in audit_files}})
    display(duplicate_summary)
    cohort = audit.loc[audit.split.eq("test") & audit.status.eq("retained")].reset_index(drop=True)
    if cohort.empty:
        raise ValueError("No PlantDoc test images survived screening.")
    cohort.to_csv(cohort_path, index=False)
    display(audit.groupby(["split","status"]).size().rename("images").reset_index())
    if MODE == "audit":
        return

    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    provenance = {
        **audit_spec,
        "inference": "RGB, PIL bilinear 224x224, ImageNet normalization, FP32",
        "runs": [{"model": model, "seed": seed, "checkpoint_path": str(weights[(model,seed)]),
                  "checkpoint_sha256": weight_hashes[(model,seed)]} for model, seed in RUNS],
        "mapping": MAPPING,
        "class_to_idx": CLASS_MAP,
        "plantdoc_source": "nirmalsankalana/plantdoc-dataset",
        "pytorch_version": str(torch.__version__),
        "device": device,
    }
    for (model, seed), path in zip(RUNS, source_paths):
        print(f"{model} seed {seed} on {len(cohort)} PlantDoc images ({device}).", flush=True)
        predict_field(weights[(model,seed)], model, seed, cohort, plantdoc, CLASS_MAP, device).to_csv(path, index=False)
    predictions = {key: read_field_prediction(path, cohort) for key, path in zip(RUNS, source_paths)}
    results = build_outputs(predictions, None, cohort, audit, CLASS_MAP, source, duplicate_summary)
    provenance["field_output_sha256"] = {
        p.name: file_sha(p) for p in source_paths + [cohort_path, *audit_files]
    }
    write_json(field_meta_path, provenance)
    display(results)
    print("PlantDoc stage complete. Keep these files for MODE='report'.")

def read_field_prediction(path, cohort):
    frame = pd.read_csv(path)
    if (frame.path.duplicated().any() or not np.array_equal(frame.path, cohort.path)
            or not np.array_equal(frame.y_true, cohort.y_true)
            or not pd.api.types.is_integer_dtype(frame.y_pred) or not frame.y_pred.between(0,37).all()):
        raise ValueError(f"Field predictions do not match the frozen cohort: {path}")
    return frame

if RUN:
    run_evaluation()


Handoff: /kaggle/input/models/astitvasrivastava05/terravista-final-handoff/other/default/1
Verified split: /kaggle/input/models/astitvasrivastava05/terravista-final-handoff/other/default/1/split (1).csv
Verified lab duplicate flags: /kaggle/input/datasets/astitvasrivastava05/test-near-duplicate-flags/test_near_duplicate_flags.csv
Screened lab images retained 5,169 of 5,469; PlantDoc images retained 241.


,domain,model,classes,images,macro_f1_mean,seed_sd,ci_low,ci_high
0,plantdoc_test,resnet50,27,241,0.282247,0.020322,0.238255,0.318217
1,plantvillage_test_same_classes_duplicate_screened,resnet50,27,5169,0.980571,0.001992,0.976999,0.983974
2,plantdoc_test,mobilenetv3_large,27,241,0.232013,0.005602,0.186598,0.267642
3,plantvillage_test_same_classes_duplicate_screened,mobilenetv3_large,27,5169,0.960502,0.011595,0.955539,0.965162


Commit the notebook and files from /kaggle/working/plantdoc-field-test
